In [14]:
from sentence_transformers import models, SentenceTransformer
import os
from accelerate import Accelerator
import json
from tqdm import tqdm

In [8]:
base_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

README.md: 0.00B [00:00, ?B/s]

In [10]:
kwargs = {}
device = Accelerator().device
ckpt_path = '../models/sentence-transformers_all-mpnet-base-v2_fmsr_ft_bs32_poolmean_pdesc0.4_positiveratio_0.5seed_42'
ckpts = [int(ckpt.split('-')[-1]) for ckpt in os.listdir(ckpt_path) if 'checkpoint-' in ckpt]
ckpt_name = 'checkpoint-' + str(max(ckpts))
ckpt_path += '/' + ckpt_name
transformer = models.Transformer(ckpt_path, model_args=kwargs)
pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode="mean")
normalize = models.Normalize()
ft_model = SentenceTransformer(modules=[transformer, pooling, normalize], device=device, trust_remote_code=True, model_kwargs=kwargs)

In [11]:
data = []
with open('adiq.jsonl', 'r') as f:
    for item in f.readlines():
        data.append(json.loads(item))

In [26]:
n_correct_base = 0
n_correct_ft = 0
for item in tqdm(data):
    question = item['question']
    options = item['options']
    correct = item['correct'].index(True)
    q_emb_base = base_model.encode(question)
    options_emb_base = base_model.encode(options)
    base_similarities = base_model.similarity(q_emb_base, options_emb_base)
    base_preds = base_similarities.argmax()

    q_emb_ft = ft_model.encode(question)
    options_emb_ft = ft_model.encode(options)
    ft_similarities = ft_model.similarity(q_emb_ft, options_emb_ft)
    ft_preds = ft_similarities.argmax()

    base_correct = int(base_preds.item() == correct)
    n_correct_base += base_correct
    ft_correct = int(ft_preds.item() == correct)
    n_correct_ft += ft_correct
print(n_correct_base / len(data))
print(n_correct_ft / len(data))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6690/6690 [06:34<00:00, 16.96it/s]

0.20672645739910314
0.15097159940209268


In [27]:
n_correct_ft

1010